# AutoGen 基础示例

在此代码示例中，您将使用 [AutoGen](https://aka.ms/ai-agents/autogen) AI 框架创建一个基础代理。

此示例的目标是向您展示在后续代码示例中实现不同代理模式时将使用的步骤。


## 导入所需的 Python 包


In [6]:
!pip install \
    "azure-ai-inference~=1.0.0b9" \
    "azure-ai-projects==1.0.0b12" \
    "azure-search-documents>=11.5.2"

  Attempting uninstall: azure-ai-projects
    Found existing installation: azure-ai-projects 2.1.0m━━━━━━━━━ 3/4 [azure-ai-projects]
    Uninstalling azure-ai-projects-2.1.0:0m╺━━━━━━━━━ 3/4 [azure-ai-projects]
      Successfully uninstalled azure-ai-projects-2.1.0━━━━━━━━ 3/4 [azure-ai-projects]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [azure-ai-projects]e-ai-projects]


In [7]:
import os
from dotenv import load_dotenv

# -----------------------------------------------------
# AutoGen 核心组件导入
# -----------------------------------------------------

# 导入 AssistantAgent，这是 Autogen 中用于创建助手或 AI 角色的核心类
from autogen_agentchat.agents import AssistantAgent 
# 导入 UserMessage，用于构造发送给模型客户端的纯用户文本消息
from autogen_core.models import UserMessage 
from azure.core.credentials import AzureKeyCredential
# 导入 AzureAIChatCompletionClient，用于连接 Azure 或兼容 OpenAI 接口的模型服务
from autogen_ext.models.openai import OpenAIChatCompletionClient 
from autogen_ext.models.azure import AzureAIChatCompletionClient
# 导入 CancellationToken，用于在异步操作中允许取消任务
from autogen_core import CancellationToken 

# 导入 TextMessage，用于构造 Agent 之间或 Agent 与用户之间的消息
from autogen_agentchat.messages import TextMessage 
# 导入 Console，用于控制台输出（虽然在这个 IPython 示例中没有直接使用）
from autogen_agentchat.ui import Console 




## 创建客户端

在本示例中，我们将使用 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) 来访问 LLM。

`model` 被定义为 `gpt-4o-mini`。尝试将模型更改为 GitHub Models 市场中可用的其他模型，看看不同的结果。

作为一个简单测试，我们将运行一个简单的提示 - `法国的首都是哪里？`


In [8]:
# -----------------------------------------------------
# 1. 初始化配置和模型客户端
# -----------------------------------------------------
from autogen_ext.models.openai import OpenAIChatCompletionClient 

load_dotenv() # 加载 .env 文件中的环境变量

# 使用通义大模型初始化 OpenAI 兼容的模型客户端
client = OpenAIChatCompletionClient(
    model="qwen-max", # 指定使用的模型ID为通义千问的qwen-max。
    # 已修改：更换为 DashScope 兼容模式的 API 基础 URL
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", 
    # 已修改：对于兼容 OpenAI API 的服务，直接将 Key 传入 api_key 参数
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # 提供模型的额外信息，指导 AutoGen 如何与模型交互
    model_info={
        "json_output": True, # 表明模型支持 JSON 格式输出。
        "structured_output": True, # 表明模型支持结构化输出。
        "function_calling": True, # 表明模型支持工具调用/Function Calling。
        "vision": True, # 表明模型支持视觉（多模态）能力。
        "family": "unknown", # 模型家族信息（这里设置为未知）。
    },
)

# 使用 Azure OpenAI 服务初始化模型客户端
# client = AzureAIChatCompletionClient(
#     model="gpt-4o-mini",
#     endpoint="https://models.inference.ai.azure.com",
#     # To authenticate with the model you will need to generate a personal access token (PAT) in your GitHub settings.
#     # Create your PAT token by following instructions here: https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens
#     credential=AzureKeyCredential(os.getenv("GITHUB_TOKEN")),
#     model_info={
#         "json_output": True,
#         "function_calling": True,
#         "vision": True,
#         "family": "unknown",
#     },
# )

# -----------------------------------------------------
# 2. 客户端测试调用
# -----------------------------------------------------

# 直接使用客户端进行一次异步调用，测试连接和模型的基本功能。
# 构造一个 UserMessage，内容是 "What is the capital of France?"
result = await client.create([UserMessage(content="What is the capital of France?", source="user")])
print(result)

finish_reason='stop' content='The capital of France is Paris.' usage=RequestUsage(prompt_tokens=15, completion_tokens=7) cached=False logprobs=None thought=None


## 定义代理

现在我们已经设置了 `client` 并确认其正常工作，接下来让我们创建一个 `AssistantAgent`。每个代理可以分配以下内容：
**name** - 一个简短的名称，用于在多代理流程中方便引用。
**model_client** - 你在前一步中创建的客户端。
**tools** - 代理可以用来完成任务的可用工具。
**system_message** - 定义任务、行为和 LLM 语气的元提示。

你可以修改 system message 来观察 LLM 的响应变化。我们将在第4课中详细讲解 `tools`。


In [9]:
# -----------------------------------------------------
# 3. 创建 Assistant Agent
# -----------------------------------------------------

# 实例化 AssistantAgent，这是 AI 助手本体。
agent = AssistantAgent(
    name="assistant",
    model_client=client,
    tools=[],
    system_message="You are a travel agent that plans great vacations",
)

## 运行代理

以下函数将运行代理。我们使用 `on_message` 方法通过新消息更新代理的状态。

在这个例子中，我们用来自用户的新消息更新状态，消息内容是 `"Plan me a great sunny vacation"`。

你可以更改消息内容，看看 LLM 会有怎样不同的响应。


In [10]:
# -----------------------------------------------------
# IPython / Jupyter 环境特定的导入
# -----------------------------------------------------

# 导入 display 和 HTML，用于在 Jupyter 或 IPython 环境中格式化和显示输出
from IPython.display import display, HTML

# -----------------------------------------------------
# 4. 异步运行函数与结果展示
# -----------------------------------------------------
async def assistant_run():
    # Define the query
    # 定义用户请求
    user_query = "Plan me a great sunny vacation"

    # Start building HTML output
    # ------------------- HTML 格式化输出 - Start -------------------
    # 开始构建用于 IPython/Jupyter 展示的 HTML 结构。
    html_output = "<div style='margin-bottom:10px'>"
    html_output += "<div style='font-weight:bold'>User:</div>"
    html_output += f"<div style='margin-left:20px'>{user_query}</div>"
    html_output += "</div>"

    # Execute the agent response
    # ------------------- Agent 执行核心 -------------------
    # 调用 agent.on_messages 方法来执行代理响应。
    response = await agent.on_messages(
        [TextMessage(content=user_query, source="user")],
        cancellation_token=CancellationToken(),
    )

    # Add agent response to HTML
    # ------------------- HTML 格式化输出 - End -------------------
    # 将代理的回复内容添加到 HTML 结构中
    html_output += "<div style='margin-bottom:20px'>"
    html_output += "<div style='font-weight:bold'>Assistant:</div>"
    html_output += f"<div style='margin-left:20px; white-space:pre-wrap'>{response.chat_message.content}</div>"
    html_output += "</div>"

    # Display formatted HTML
    # 使用 IPython 的 display(HTML(...)) 函数展示最终格式化的结果。
    display(HTML(html_output))

# Run the function
await assistant_run()


---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
